# 6장 실습: Python에서 Neo4j 연결하기

실습 가이드 6장 6-1 ~ 6-3의 코드를 순서대로 실행하는 노트북입니다. 설명은 실습 가이드를 함께 봅니다.

| 항목 | 내용 |
|---|---|
| 연결할 DB | Recommendations (4장과 같은 DB. Docker `bolt://localhost:7687` 또는 Recommendations 샌드박스) |
| .env 설정 | Recommendations DB의 NEO4J 네 줄 (실습 가이드 5-5) |

.env를 바꿨다면 노트북 상단의 `Restart`로 커널을 재시작한 뒤 처음부터 실행합니다. 연결 셀의 `connect_graph()`는 .env를 다시 읽지만, 앞 셀에서 만든 객체가 이전 DB를 가리킬 수 있으므로 처음부터 다시 실행합니다.

## 1. 환경 확인 (실습 가이드 6-1)

In [1]:
# 설치된 버전을 확인한다. 강사 화면과 다르면 uv sync를 다시 실행한다.
from tools.graphrag_tools import check_env

# 커널 경로(.venv), 작업 위치(프로젝트 루트), 패키지 버전을 출력한다.
check_env(["python-dotenv", "neo4j", "langchain-neo4j"])

Python : c:\study\2026-graph-rag\.venv\Scripts\python.exe
작업 위치: c:\study\2026-graph-rag
python-dotenv              1.2.4
neo4j                      6.4.1
langchain-neo4j            0.11.0


## 연결 확인

`Neo4jGraph`는 연결할 때 DB 스키마를 읽습니다. 이때 APOC 플러그인을 쓰므로, APOC가 없는 DB에서는 `Could not use APOC procedures` 오류가 납니다(Sandbox, AuraDB, 실습용 Docker 이미지에는 들어 있습니다).

출력된 레이블별 노드 수로 이 차시에 맞는 DB에 연결됐는지 확인합니다.

In [2]:
from tools.graphrag_tools import connect_graph

# .env의 접속 정보로 Neo4jGraph를 만들고 레이블별 노드 수를 출력한다(실습 가이드 6-2).
# 함수 안에서 하는 일: .env의 NEO4J 네 값을 딕셔너리 NEO4J로 묶고 Neo4jGraph(**NEO4J)로 연결한다.
# expect의 레이블이 없으면 다른 DB에 연결됐다고 알려 준다.
# 연결 오류는 이 줄에서 난다. ServiceUnavailable: 서버가 꺼져 있거나 시작 중 / AuthError: 비밀번호 / APOC 오류: 플러그인 없음
graph, NEO4J = connect_graph(expect=["Movie", "User", "Genre"])

연결: bolt://localhost:7687 | DB: neo4j
{'label': 'Actor', 'n': 15443}
{'label': 'Movie', 'n': 9125}
{'label': 'Director', 'n': 3604}
{'label': 'User', 'n': 671}
{'label': 'Genre', 'n': 20}


## 2. 스키마 확인 (실습 가이드 6-2)

`graph.schema`는 노드 속성, 관계 속성, 관계 패턴을 문자열로 정리한 것입니다. 3장 `CALL db.schema.visualization`이 그림으로 보여 준 구조를 텍스트로 나타낸 것이고, 8장에 LLM에게 넘기는 스키마가 이 문자열입니다.

In [3]:
# query()는 Cypher 문자열을 실행하고 결과를 딕셔너리 리스트로 돌려준다.
print(graph.query("RETURN 'connected' AS status"))
# 연결할 때 읽어 둔 스키마 문자열. 8장 Text2Cypher에서 이 문자열을 LLM 프롬프트에 그대로 넣는다.
print(graph.schema)

[{'status': 'connected'}]
Node properties:
Movie {url: STRING, runtime: INTEGER, revenue: INTEGER, budget: INTEGER, embedding: LIST, imdbRating: FLOAT, released: STRING, countries: LIST, languages: LIST, plot: STRING, imdbVotes: INTEGER, imdbId: STRING, year: INTEGER, poster: STRING, movieId: STRING, tmdbId: STRING, title: STRING}
Genre {name: STRING}
User {userId: STRING, name: STRING}
Actor {url: STRING, bornIn: STRING, bio: STRING, died: DATE, born: DATE, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING}
Director {url: STRING, bornIn: STRING, born: DATE, died: DATE, tmdbId: STRING, imdbId: STRING, name: STRING, poster: STRING, bio: STRING}
Person {url: STRING, died: DATE, bornIn: STRING, born: DATE, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING, bio: STRING}
Relationship properties:
RATED {rating: FLOAT, timestamp: INTEGER}
ACTED_IN {role: STRING}
DIRECTED {role: STRING}
The relationships:
(:Movie)-[:IN_GENRE]->(:Genre)
(:User)-[:RATED]->(:Movie)
(:Actor)-

## 3. Browser 쿼리를 Python에서 실행 (실습 가이드 6-3)

Browser에 입력한 쿼리를 `graph.query()`에 문자열로 넣으면 그대로 실행됩니다. 결과는 딕셔너리의 리스트이고, `RETURN`에서 `AS`로 붙인 이름이 키가 됩니다.

Recommendations의 개봉 연도는 `year`(숫자)입니다. `released`는 `'1995-11-22'` 같은 문자열이라 `released >= 2000`으로 비교하면 결과가 0행입니다(실습 가이드 4-1 "Movies와 다른 점").

In [4]:
# Recommendations의 개봉 연도는 year(숫자)다. released는 '1995-11-22' 같은 문자열이라 2000과 비교하면 0행이 된다.
# Browser에 입력한 쿼리를 문자열로 그대로 넘긴다. 여러 줄 쿼리는 삼중 따옴표로 감싼다.
rows = graph.query("""
MATCH (m:Movie)
WHERE m.year >= 2000
RETURN m.title AS title, m.year AS year
ORDER BY m.year DESC
LIMIT 5
""")
# 각 행은 딕셔너리이고 키는 RETURN 절의 AS 이름이다. AS를 빼면 키가 'm.title'처럼 식 전체가 된다.
for row in rows:
    print(row["title"], row["year"])

Shin Godzilla 2016
Mohenjo Daro 2016
Rustom 2016
Sharknado 4: The 4th Awakens 2016
Hell or High Water 2016


## 실습 과제

위의 셀을 모두 실행한 뒤 진행합니다. 과제마다 문제, 정답 코드, 해설 순서로 되어 있습니다. 먼저 직접 작성해 본 뒤 정답 코드와 비교합니다.

### 과제 1. 3장 쿼리를 Python에서 실행

3장 실습 과제의 쿼리 세 개(Tom Hanks 출연작, 2000년 이후 개봉작 10편, Keanu Reeves 공동 출연자)를 Recommendations DB에 맞게 고쳐 `graph.query()`로 실행하고, 결과를 한 줄에 한 건씩 출력하세요. 개봉 연도는 `released` 대신 `year`를 씁니다. 결과 수가 3장(Movies)과 어떻게 다른지도 확인합니다.

In [5]:
# 과제 1 정답
# 쿼리 이름과 Cypher를 딕셔너리로 묶어 반복문 하나로 실행한다.
queries = {
    "Tom Hanks 출연작": """
        MATCH (p:Person {name: "Tom Hanks"})-[:ACTED_IN]->(m:Movie)
        RETURN m.title AS title
        ORDER BY title""",
    "2000년 이후 개봉작": """
        MATCH (m:Movie)
        WHERE m.year >= 2000
        RETURN m.title AS title, m.year AS year
        ORDER BY year, title
        LIMIT 10""",
    "Keanu Reeves 공동 출연자": """
        MATCH (k:Person {name: "Keanu Reeves"})-[:ACTED_IN]->(:Movie)<-[:ACTED_IN]-(co:Person)
        RETURN DISTINCT co.name AS coActor
        ORDER BY coActor""",
}

# 건수를 함께 출력하면 3장(Movies) 결과와 비교하기 쉽다.
# Actor 노드는 Person 레이블도 함께 가지고 있어서 (p:Person) 패턴이 Recommendations에서도 그대로 동작한다.
for name, q in queries.items():
    rows = graph.query(q)
    print(f"== {name} ({len(rows)}건)")
    for row in rows:
        print(row)

== Tom Hanks 출연작 (38건)
{'title': "'burbs, The"}
{'title': 'Angels & Demons'}
{'title': 'Apollo 13'}
{'title': 'Bachelor Party'}
{'title': 'Big'}
{'title': 'Bonfire of the Vanities'}
{'title': 'Captain Phillips'}
{'title': 'Catch Me If You Can'}
{'title': "Charlie Wilson's War"}
{'title': 'Cloud Atlas'}
{'title': 'Da Vinci Code, The'}
{'title': 'Dragnet'}
{'title': 'Extremely Loud and Incredibly Close'}
{'title': 'Forrest Gump'}
{'title': 'From the Earth to the Moon'}
{'title': 'Green Mile, The'}
{'title': 'Joe Versus the Volcano'}
{'title': 'Ladykillers, The'}
{'title': 'Larry Crowne'}
{'title': 'League of Their Own, A'}
{'title': 'Man with One Red Shoe, The'}
{'title': 'Money Pit, The'}
{'title': 'Nothing in Common'}
{'title': 'Philadelphia'}
{'title': 'Polar Express, The'}
{'title': 'Punchline'}
{'title': 'Saving Mr. Banks'}
{'title': 'Saving Private Ryan'}
{'title': 'Sleepless in Seattle'}
{'title': 'Splash'}
{'title': 'Terminal, The'}
{'title': 'Toy Story'}
{'title': 'Toy Story 2'}

#### 과제 1 해설

실행 결과 (Recommendations, Docker Neo4j 5.26): Tom Hanks 출연작 38편, 2000년 이후 개봉작 10편(조건에 맞는 영화는 모두 3,898편), Keanu Reeves 공동 출연자 94명입니다. 3장 Movies 결과(12편, 14명)보다 많은 것은 Recommendations에 영화가 9,125편 들어 있기 때문입니다.

- `graph.query()`의 결과는 딕셔너리의 리스트입니다. 키는 `RETURN`에서 `AS`로 붙인 이름(`title`, `year`, `coActor`)이고, `AS`가 없으면 `m.title`처럼 표현식이 그대로 키가 됩니다.
- Recommendations의 배우는 `Actor` 레이블과 `Person` 레이블을 함께 가집니다. 그래서 3장의 `(p:Person {name: "Tom Hanks"})` 패턴을 고치지 않아도 됩니다.
- 개봉 연도는 `year`(숫자)입니다. `released`는 문자열이라 `>= 2000`이 항상 거짓이 되어 오류 없이 0행이 나옵니다. `year`가 없는 영화 29편은 `WHERE`에서 빠집니다.
- 2000년에 개봉한 영화만 수백 편이라 `ORDER BY year`만 쓰면 어떤 10편이 나올지 정해지지 않습니다. 두 번째 정렬 기준 `title`을 넣는 이유입니다.
- 공동 출연자 쿼리에서 `DISTINCT`를 빼면 94명이 아니라 99행이 나옵니다. Matrix 시리즈처럼 여러 편에 함께 나온 배우가 영화마다 한 번씩 반복되기 때문입니다.
- 쿼리를 딕셔너리에 담아 반복 실행하는 방식은 8장에 LLM이 만든 쿼리를 실행할 때도 그대로 씁니다.

### 과제 2. 스키마에서 Movie 속성 찾기

`graph.schema` 출력에서 Movie 노드의 속성 목록을 찾으세요. 문자열에서 찾는 방법과, 구조화된 스키마(`graph.structured_schema`)에서 꺼내는 방법을 모두 해 봅니다.

In [6]:
# 과제 2 정답
# 방법 1: 스키마 문자열에서 Movie 줄 찾기
for line in graph.schema.splitlines():
    if line.startswith("Movie "):
        print(line)

# 방법 2: 구조화된 스키마(딕셔너리)에서 꺼내기
# schema는 사람이 읽는 문자열, structured_schema는 같은 내용을 코드에서 꺼내 쓰기 좋게 담은 딕셔너리다.
for prop in graph.structured_schema["node_props"]["Movie"]:
    print(prop)

Movie {url: STRING, runtime: INTEGER, revenue: INTEGER, budget: INTEGER, embedding: LIST, imdbRating: FLOAT, released: STRING, countries: LIST, languages: LIST, plot: STRING, imdbVotes: INTEGER, imdbId: STRING, year: INTEGER, poster: STRING, movieId: STRING, tmdbId: STRING, title: STRING}
{'property': 'url', 'type': 'STRING'}
{'property': 'runtime', 'type': 'INTEGER'}
{'property': 'revenue', 'type': 'INTEGER'}
{'property': 'budget', 'type': 'INTEGER'}
{'property': 'embedding', 'type': 'LIST'}
{'property': 'imdbRating', 'type': 'FLOAT'}
{'property': 'released', 'type': 'STRING'}
{'property': 'countries', 'type': 'LIST'}
{'property': 'languages', 'type': 'LIST'}
{'property': 'plot', 'type': 'STRING'}
{'property': 'imdbVotes', 'type': 'INTEGER'}
{'property': 'imdbId', 'type': 'STRING'}
{'property': 'year', 'type': 'INTEGER'}
{'property': 'poster', 'type': 'STRING'}
{'property': 'movieId', 'type': 'STRING'}
{'property': 'tmdbId', 'type': 'STRING'}
{'property': 'title', 'type': 'STRING'}


#### 과제 2 해설

실행 결과 (Recommendations): `Movie {url, runtime, revenue, imdbRating, released: STRING, countries, languages, plot, imdbVotes, imdbId, year: INTEGER, poster, movieId, tmdbId, title, embedding, budget}`. Movie 노드의 속성은 17개입니다. 3장 Movies의 Movie(tagline, title, released 3개)보다 훨씬 많고, 이름이 같은 `released`도 자료형이 다릅니다(Movies는 INTEGER, Recommendations는 STRING). `embedding`은 9장에서 줄거리 임베딩을 만든 DB에만 보입니다.

| 방법 | 형태 | 쓰는 곳 |
|---|---|---|
| `graph.schema` | 사람이 읽는 문자열 | 8장 Text2Cypher 프롬프트에 그대로 넣음 |
| `graph.structured_schema["node_props"]["Movie"]` | `{'property': ..., 'type': ...}` 딕셔너리의 리스트 | 코드에서 속성 이름을 골라 쓸 때 |

- 3장 `CALL db.schema.visualization`은 레이블과 관계만 그림으로 보여 주고 속성은 보여 주지 않습니다. 속성과 자료형까지 보려면 `graph.schema`를 씁니다.
- `structured_schema`에는 `node_props` 외에 관계 속성(`rel_props`)과 관계 패턴(`relationships`)도 들어 있습니다.